# Exemple de stacking 

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
import pandas as pd

# Utilitaires (identique aux contenus du git dans src de la branche main)

ANTENNA_NAMES = ["041A9F60", "041A9F61", "0F173B12", "0F173B13",
                 "PORT_1", "PORT_2", "PORT_3", "PORT_4"]
ANTENNA_POSITIONS = {
    "041A9F60": (2.8, 2.2, 3.6),
    "041A9F61": (2.7, 3.8, 3.6),
    "PORT_1":   (5.7, 4.0, 3.9),
    "PORT_2":   (8.5, 3.8, 3.9),
    "PORT_3":   (5.6, 1.7, 3.9),
    "PORT_4":   (8.4, 2.1, 3.9),
    "0F173B12": (11.0, 1.7, 4.1),
    "0F173B13": (11.2, 4.0, 4.3),
}



# === Loader ===

META_COLS = ["EPC", "xyz", "x", "y", "z", "polar", "nearestAnt", "nearestD"]

class TxRxDsLoader:
    """
    Loader for ds_Tx_Rx.csv.
    
    Features columns: max__rssi__* and max__Delta_rssi__*
    Distances columns: <antenna>__NONE  (e.g., PORT_1__NONE, 041A9F60__NONE)
    Missing RSSI values replaced with -130 dBm.
    """
    def __init__(self):
        self.feature_cols:  list[str] = []
        self.antenna_names: list[str] = []

    def load(self, path):
        df = pd.read_csv(path)

        dist_cols = [
            c for c in df.columns
            if c not in META_COLS
            and c.endswith("__NONE")
            and not c.startswith("max__")
        ]
        self.antenna_names = [c.replace("__NONE", "") for c in dist_cols]

        feat_cols = [
            c for c in df.columns
            if c not in META_COLS and c not in dist_cols
        ]
        self.feature_cols = feat_cols

        ds = df.copy()
        ds[feat_cols] = ds[feat_cols].fillna(-130)

        df_clean = ds[feat_cols + ["x", "y", "z"] + dist_cols].reset_index(drop=True)

        X           = df_clean[feat_cols].values.astype(np.float32)
        y_xyz       = df_clean[["x", "y", "z"]].values.astype(np.float32)
        y_distances = df_clean[dist_cols].values.astype(np.float32)

        return X, y_xyz, y_distances, df_clean



# === cross validation === 

def cross_validate(X, y, train_fn, predict_fn, eval_fn, aggregate_fn, k=5, random_state=42):
    """
    Perform k-fold cross-validation and return detailed results per fold and aggregated summary.
    
    Args:
        X: Input features
        y: Target values
        train_fn: Function to train a model ((X_train, y_train) -> model)
        predict_fn: Function to make predictions ((model, X_test) -> y_pred))
        eval_fn: Function to evaluate predictions ((y_test, y_pred, test_index) -> dict[str, float])
        aggregate_fn: Function to aggregate fold results into summary statistics
            ((List[Dict[str, float]]) -> Dict[str, Dict[str, float]])
        k: Number of folds
        random_state: Random seed for reproductibility
    Returns:
        Dict with two keys:
        - "folds": List of dictionaries, each containing metrics for a fold.
        - "summary": Aggregated summary of metrics across all folds.
    """
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    
    folds_results = []

    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        model = train_fn(X_train, y_train)
        y_pred = predict_fn(model, X_test)

        fold_result = eval_fn(y_test, y_pred, test_index) # Passes test_index to allow eval_fn to access other data
        folds_results.append(fold_result)
        summary_result = aggregate_fn(folds_results)
        
    return {
        "folds": folds_results,
        "summary": summary_result
    }
    


# === eval_fn and aggregate_fn ===

def eval_fn(y_true, y_pred, test_index):
    errs = np.linalg.norm(y_pred - y_true, axis=1)
    return {
        "mae":    mae_3d(errs),
        "3Derror": rmse_3d(errs),
        "rmse":   rmse_xyz(y_true, y_pred),
        "acc_05": threshold_accuracy(errs, 0.5),
        "acc_1":  threshold_accuracy(errs, 1.0),
        "acc_2":  threshold_accuracy(errs, 2.0),
        "acc_3":  threshold_accuracy(errs, 3.0),
    }

def aggregate_fn(folds_results):
    summary = {}
    for key in folds_results[0]:
        vals = np.array([f[key] for f in folds_results])
        summary[key] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
    return summary



# === Métriques ===

def mae_3d(errs: np.ndarray) -> float:
    """
    Calculate the Mean Absolute Error (MAE) from an array of errors.

    This function works for:
    - distance errors between predicted and true antenna distances
    - Euclidean position errors in 3D space

    Args:
        errs: Array of errors (absolute distance errors or Euclidean errors)

    Returns:
        Mean error.
    """
    return float(np.mean(errs))

def rmse_3d(errs: np.ndarray) -> float:
    """Calculate the 3D Root Mean Squared Error (RMSE) between true and predicted positions.

    The 3D RMSE is the square root of the average squared Euclidean distances between each pair
    of true and predicted 3D coordinates.

    Args: 
        errs: Euclidean distance errors for each sample
    Returns:
        Root Mean Squared Error of the Euclidean distances between predicted and true 3D positions
    """
    return np.sqrt(np.mean(errs ** 2))

def rmse_xyz(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Per-axis average RMSE over x, y, z coordinates.

    Equivalent to sklearn's sqrt(mean_squared_error(y_true, y_pred)) on (n, 3) arrays,
    which averages the squared errors over all samples AND all 3 axes before taking sqrt.

    This differs from rmse_3d by a factor of 1/√3:
        rmse_xyz = rmse_3d(errs) / √3

    Args:
        y_true: True positions, shape (n, 3)
        y_pred: Predicted positions, shape (n, 3)
    Returns:
        Per-axis RMSE (in meters). Smaller than rmse_3d for the same predictions.
    """
    diff = np.asarray(y_true) - np.asarray(y_pred)
    return float(np.sqrt(np.mean(diff ** 2)))

def threshold_accuracy(errs: np.ndarray, threshold: float) -> float:
    """Calculate the percentage of predictions with error below a given threshold.

    Args:
        errs: Euclidean distance errors for each sample
    Returns:
        Percentage of predictions with error ≤ threshold
    """
    return (errs <= threshold).mean()*100



# === print_summary ===

def print_summary(label, summary):
    m = summary
    print(
        f"{label:<35} "
        f"RMSE={m['rmse']['mean']:.3f}±{m['rmse']['std']:.3f}  "
        f"MAE={m['mae']['mean']:.3f}±{m['mae']['std']:.3f}  "
        f"3Derror={m['3Derror']['mean']:.3f}±{m['3Derror']['std']:.3f}  "
        f"≤0.5m={m['acc_05']['mean']:.1f}%  "
        f"≤1m={m['acc_1']['mean']:.1f}%  "
        f"≤2m={m['acc_2']['mean']:.1f}%  "
        f"≤3m={m['acc_3']['mean']:.1f}%"
    )

In [2]:
# Charger les données

print(f"Loading dataset: {"ds_Tx_Rx.csv"}")
loader = TxRxDsLoader()
X, y_xyz, y_distances, df_clean = loader.load("data/ds_Tx_Rx.csv")

results_cv = {}

Loading dataset: ds_Tx_Rx.csv


In [3]:
# ExtraTrees MultiOutput (baseline)

import time
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.multioutput import MultiOutputRegressor, RegressorChain

_params = dict(n_estimators=200, random_state=0, n_jobs=-1)

def train_et_mo(X_train, y_train):
    m = MultiOutputRegressor(ExtraTreesRegressor(**_params), n_jobs=-1)
    m.fit(X_train, y_train)
    return m

_t0 = time.time()
results_cv["ET_MO"] = cross_validate(
    X, y_xyz,
    train_fn=train_et_mo,
    predict_fn=lambda m, X: m.predict(X),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution : {time.time() - _t0:.1f}s")
print_summary("ExtraTrees MultiOutput", results_cv["ET_MO"]["summary"])


Temps d'exécution : 171.9s
ExtraTrees MultiOutput              MAE=1.099±0.014  3Derror=1.242±0.020  RMSE=0.717±0.011  ≤0.5m=10.6%  ≤1m=50.1%  ≤2m=93.4%  ≤3m=99.0%


In [ ]:
# Stacking avec écart-type des arbres comme signal d'incertitude pour le blender

# ~ 58 minutes (12 workers CPU)

from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
from sklearn.linear_model import Ridge

def et_predict_with_variance(model_mo, X):
    """
    Retourne (mean, std) pour un MultiOutputRegressor(ExtraTreesRegressor).
    Chaque estimateur interne est un ExtraTrees avec n arbres :
    on récupère la prédiction de chaque arbre et on calcule mean/std.

    Retourne :
        means : shape (n, 3)   - prédiction moyenne par coordonnée
        stds  : shape (n, 3)   - écart-type entre arbres par coordonnée
    """
    per_output = []
    for estimator in model_mo.estimators_:          # 1 ET par coordonnée (x, y, z)
        # estimator.estimators_ = liste des arbres individuels
        tree_preds = np.array([
            tree.predict(X) for tree in estimator.estimators_
        ])  # shape (n_trees, n_samples)
        per_output.append(tree_preds)

    # per_output : liste de 3 tableaux (n_trees, n_samples)
    means = np.stack([p.mean(axis=0) for p in per_output], axis=1)  # (n, 3)
    stds  = np.stack([p.std(axis=0)  for p in per_output], axis=1)  # (n, 3)
    return means, stds

def _chain_et_std(chain, X):
    stds = []
    X_aug = X.copy()
    for est in chain.estimators_:
        tree_preds = np.array([t.predict(X_aug) for t in est.estimators_])
        stds.append(tree_preds.std(axis=0))
        X_aug = np.hstack([X_aug, tree_preds.mean(axis=0).reshape(-1, 1)])
    return np.stack(stds, axis=1)  # (n, 3)

class KFoldStacker:
    """
    Stacking avec méta-features enrichies :
      - prédictions (x,y,z) des modèles de base
      - std des arbres ET MultiOutput            , 3 features (incertitude)
      - std des arbres Chain ET                  , 3 features (incertitude)
    Total méta-features : 18
    Blender : Ridge
    """
    def __init__(self, n_splits=5, random_state=0):
        self.n_splits     = n_splits
        self.random_state = random_state
        self.base_models_ = None
        self.blender_     = None

    def _make_bases(self):
        return [
            ("ET_MO",    MultiOutputRegressor(ExtraTreesRegressor(n_estimators=300, random_state=0, n_jobs=-1),n_jobs=-1)),
            
            ("ET_direct", ExtraTreesRegressor(n_estimators=300, random_state=0, n_jobs=-1)),

            ("Chain_ET", RegressorChain(ExtraTreesRegressor(n_estimators=300, random_state=0, n_jobs=-1),order=[0, 1, 2])),

            ("XGB_MO",   MultiOutputRegressor(
                             xgb.XGBRegressor(n_estimators=400, max_depth=6,
                                              learning_rate=0.05, subsample=0.8,
                                              colsample_bytree=0.8, n_jobs=-1,
                                              random_state=0, verbosity=0),
                             n_jobs=-1)),
            
            ("KNN",      MultiOutputRegressor(KNeighborsRegressor(n_neighbors=7, weights="distance", n_jobs=-1),n_jobs=-1)),
        ]

    def _build_meta(self, models_dict, X):
        parts = []

        for name, model in models_dict.items():
            preds = model.predict(X)
            if preds.ndim == 1:
                preds = preds.reshape(-1, 1)
            parts.append(preds)  # toujours les prédictions

            # Ajout de l'incertitude pour les modèles qui la supportent
            if name == "ET_MO":
                _, std = et_predict_with_variance(model, X)
                parts.append(std)
            elif name == "Chain_ET":
                parts.append(_chain_et_std(model, X))  # fonction extraite

        return np.hstack(parts)

    def fit(self, X, y):
        n = X.shape[0]
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)

        meta_parts = []  # on accumule les lignes dans une liste

        for train_idx, val_idx in kf.split(X):
            fold_models = {}
            for name, model in self._make_bases():
                model.fit(X[train_idx], y[train_idx])
                fold_models[name] = model

            meta_val = self._build_meta(fold_models, X[val_idx])  # (n_val, ?)
            meta_parts.append((val_idx, meta_val))

        # Reconstruction ordonnée dans meta_X, maintenant qu'on connaît n_meta_features
        n_meta_features = meta_parts[0][1].shape[1]
        meta_X = np.empty((n, n_meta_features))
        for val_idx, meta_val in meta_parts:
            meta_X[val_idx] = meta_val

        # Ré-entraîner sur tout X
        self.base_models_ = {}
        for name, model in self._make_bases():
            model.fit(X, y)
            self.base_models_[name] = model

        self.blender_ = MultiOutputRegressor(Ridge(alpha=1.0))
        self.blender_.fit(meta_X, y)
        return self

    def predict(self, X):
        meta_X = self._build_meta(self.base_models_, X)
        return self.blender_.predict(meta_X)


results_cv["Stacking"] = cross_validate(
    X, y_xyz,
    train_fn=lambda Xtr, ytr: KFoldStacker().fit(Xtr, ytr),
    predict_fn=lambda m, Xt: m.predict(Xt),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print_summary("Stacking",
              results_cv["Stacking"]["summary"])



Stacking                            MAE=1.074±0.010  3Derror=1.223±0.015  RMSE=0.706±0.009  ≤0.5m=12.9%  ≤1m=51.7%  ≤2m=93.1%  ≤3m=98.9%


In [7]:
# TABLEAU RECAPITULATIF

SEP = "=" * 130
print(f"\n\n{SEP}")
print("RECAPITULATIF (cross-validation 5 folds)")
print(SEP)
header = (
    f"{'Approche':<40} "
    f"{'RMSE':>12} "
    f"{'MAE':>12} "
    f"{'3Derror':>12} "
    f"{'≤0.5m':>8} "
    f"{'≤1m':>8} "
    f"{'≤2m':>8} "
    f"{'≤3m':>8}"
)
print(header)
print("-" * 130)

# Trier les résultats par RMSE (moyenne, croissant)
sorted_results = sorted(
    results_cv.items(),
    key=lambda x: x[1]["summary"]["rmse"]["mean"]
)

for label, res in sorted_results:
    s = res["summary"]
    print(
        f"{label:<40} "
        f"{s['rmse']['mean']:>6.3f}±{s['rmse']['std']:.3f}  "
        f"{s['mae']['mean']:>6.3f}±{s['mae']['std']:.3f}  "
        f"{s['3Derror']['mean']:>6.3f}±{s['3Derror']['std']:.3f}  "
        f"{s['acc_05']['mean']:>5.1f}%  "
        f"{s['acc_1']['mean']:>5.1f}%  "
        f"{s['acc_2']['mean']:>5.1f}%  "
        f"{s['acc_3']['mean']:>5.1f}%"
    )
print(SEP)



RECAPITULATIF (cross-validation 5 folds)
Approche                                         RMSE          MAE      3Derror    ≤0.5m      ≤1m      ≤2m      ≤3m
----------------------------------------------------------------------------------------------------------------------------------
Stacking                                  0.706±0.009   1.074±0.010   1.223±0.015   12.9%   51.7%   93.1%   98.9%
ET_MO                                     0.717±0.011   1.099±0.014   1.242±0.020   10.6%   50.1%   93.4%   99.0%
